# Dataset Analysis: Fake Instagram Profile Detector

This notebook performs Exploratory Data Analysis (EDA) on the Hugging Face Instagram Fake Profile Detection dataset (`nahiar/instagram_bot_detection`). The goal is to understand the dataset structure, identify any data quality issues, and uncover patterns that distinguish fake profiles from real ones.

We will follow a structured approach to evaluate the dataset.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

# Ensure directories exist
os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../outputs/plots', exist_ok=True)

### 1. Load the CSV

First, we will load the dataset. If the dataset does not already exist locally, we will attempt to download it using the `datasets` library from Hugging Face.

In [ ]:
csv_path = '../data/raw/instagram_profiles.csv'

if not os.path.exists(csv_path):
    print("Downloading dataset...")
    try:
        ds = load_dataset("nahiar/instagram_bot_detection")
        df = ds['train'].to_pandas()
        df.to_csv(csv_path, index=False)
        print("SUCCESS: Downloaded via datasets library")
    except Exception as e:
        print(f"FAILED: {e}")
        
df = pd.read_csv(csv_path)

### 2. Display the first 5 rows

Let's look at a preview of our dataset to understand the features and sample values.

In [ ]:
df.head()

### 3. Check shape

We need to confirm the dataset contains approximately 5,000 rows and the 12 expected columns (11 features + 1 target).

In [ ]:
df.shape

### 4. Check column names

We'll print the exact column names to verify if they match our feature dictionary. Any discrepancies in spelling, symbols (like `#` or `==`), or spacing should be noted here.

In [ ]:
df.columns.tolist()

### 5. Check data types

Checking the inferred data types ensures numerical features were loaded properly and weren't incorrectly read as string objects.

In [ ]:
df.dtypes

### 6. Check missing values

Although the dataset is documented to have no missing values, it's best practice to verify this programmatically.

In [ ]:
print(df.isnull().sum())
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

### 7. Check duplicate rows

We check if there are perfectly identical rows in the dataset. If we find duplicates, we will save a deduplicated version for any subsequent training steps.

In [ ]:
duplicates = df.duplicated().sum()
print(f"Total duplicate rows: {duplicates}")

if duplicates > 0:
    display(df[df.duplicated(keep=False)].sort_values(by=df.columns.tolist()).head())
    df = df.drop_duplicates()
    processed_path = '../data/processed/instagram_profiles_clean.csv'
    df.to_csv(processed_path, index=False)
    print(f"Saved deduplicated data to {processed_path}")

### 8. Class distribution

The dataset documentation states a 50/50 balance of fake and real profiles. Let's confirm this using the `fake` target variable.

In [ ]:
print("Counts:")
print(df['fake'].value_counts())
print("\nPercentages:")
print(df['fake'].value_counts(normalize=True) * 100)

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='fake')
plt.title('Class Distribution (Fake vs Real)')
plt.savefig('../outputs/plots/class_distribution.png')
plt.show()

### 9. Summary statistics

We compute summary statistics for all numeric columns to understand scales, averages, and variability.

In [ ]:
df.describe()

### 10. Feature distributions

We visualize the distribution of each feature. We'll use histograms for continuous/count features and bar charts for binary features.

In [ ]:
numeric_features = ['#posts', '#followers', '#follows', 'description length', 'nums/length username', 'nums/length fullname', 'fullname words']
binary_features = ['profile pic', 'private', 'external URL', 'name==username']

for feat in numeric_features:
    if feat in df.columns:
        plt.figure(figsize=(8, 4))
        sns.histplot(df[feat], bins=50, kde=True)
        plt.title(f'Distribution of {feat}')
        safe_name = feat.replace('/', '_').replace('#', 'num_').replace('==', '_eq_').replace(' ', '_')
        plt.savefig(f'../outputs/plots/dist_{safe_name}.png')
        plt.show()
        
for feat in binary_features:
    if feat in df.columns:
        plt.figure(figsize=(6, 4))
        sns.countplot(data=df, x=feat)
        plt.title(f'Count of {feat}')
        safe_name = feat.replace('==', '_eq_').replace(' ', '_')
        plt.savefig(f'../outputs/plots/count_{safe_name}.png')
        plt.show()

### 11. Fake vs. Real comparison

Now we overlay the target label on our numeric feature distributions. This helps us visually spot which features separate fake profiles from real ones (e.g., lower post counts for fake profiles).

In [ ]:
for feat in numeric_features:
    if feat in df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(data=df, x='fake', y=feat)
        plt.title(f'{feat} by Class (Fake vs Real)')
        safe_name = feat.replace('/', '_').replace('#', 'num_').replace('==', '_eq_').replace(' ', '_')
        plt.savefig(f'../outputs/plots/boxplot_{safe_name}.png')
        plt.show()

### 12. Correlation analysis

A correlation heatmap will show relationships among features, and more importantly, how strongly each feature correlates with the `fake` target variable.

In [ ]:
plt.figure(figsize=(10, 8))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig('../outputs/plots/correlation_heatmap.png')
plt.show()

correlations = corr['fake'].sort_values(ascending=False)
print("Correlations with target 'fake':")
print(correlations)

**Key Correlation Findings:**
- Features like `profile pic`, `description length`, and `#posts` typically have strong negative correlations with the `fake` label, meaning fake profiles often lack pictures, have shorter bios, and fewer posts.
- Features like `nums/length username` might show positive correlation with the `fake` label.

### 13. Outlier check

We perform a brief outlier check using the Interquartile Range (IQR) method on `#followers`, `#follows`, and `#posts`. We don't remove them yet, but it's important to document their existence for future modeling steps.

In [ ]:
for feat in ['#followers', '#follows', '#posts']:
    if feat in df.columns:
        Q1 = df[feat].quantile(0.25)
        Q3 = df[feat].quantile(0.75)
        IQR = Q3 - Q1
        outliers = df[(df[feat] < (Q1 - 1.5 * IQR)) | (df[feat] > (Q3 + 1.5 * IQR))]
        print(f"{feat}: {len(outliers)} outliers")